# AiXbio Hackathon  ESM-2 Toxin Screener Demo

This interactive notebook demonstrates our 88% accurate, zero-shot toxin screening tool against ProteinMPNN redesigns.
It uses a mechanistic interpretability approach (Direct Logit Attribution & Sparse Autoencoders) to explain *why* a sequence is flagged.

In [ ]:
# ── 0. Install Dependencies ──────────────────────────────────────────────
# Run this cell once to install required packages
!pip install -q torch transformers biopython scikit-learn matplotlib numpy




In [ ]:
# Optional: Install interPLM for Sparse Autoencoder (SAE) Toxin Motif Extraction
!git clone https://github.com/ElanaPearl/interPLM.git && pip install -e interPLM


In [ ]:
# ── 0b. Download Probe Weights & Screen Script ─────────────────────────
# Run this cell if you are running in Google Colab or don't have the files locally.
import os
import urllib.request

REPO_URL = "https://raw.githubusercontent.com/punctualprocrastinator/cautious-goggles/main"

os.makedirs("results", exist_ok=True)

files_to_download = {
    "screen.py": f"{REPO_URL}/screen.py",
    "results/probe_direction.npy": f"{REPO_URL}/results/probe_direction.npy"
}

for filepath, url in files_to_download.items():
    if not os.path.exists(filepath):
        print(f"Downloading {filepath}...")
        try:
            urllib.request.urlretrieve(url, filepath)
            print(f"Successfully downloaded {filepath}")
        except Exception as e:
            print(f"⚠️ Error downloading {filepath}: {e}\nPlease update REPO_URL.")


In [ ]:
#  1. Setup and Load Model 
import torch, json, numpy as np, matplotlib.pyplot as plt
from screen import load_model, score_seq, risk_level, explain, FEATURE_NAMES

print("Loading ESM-2 and Toxin Circuit Probe...")
tok, esm2, probe, probe_dir_np, sae = load_model()
print("Ready.")

### Test a Sequence
Try pasting a known toxin, a random protein, or an AI redesign below.

In [ ]:
#  2. Run Toxin Screen 

TEST_SEQUENCE = "MKTIIALSYIFCLVFADYKDDDDKGAPSITCPPCGVKRNDGCCPIVCPPGGCPKCPRC"
# (This is a sample toxin snippet. Replace it with any sequence!)

score = score_seq(TEST_SEQUENCE, tok, esm2, probe)
level = risk_level(score)

print(f"Sequence Length: {len(TEST_SEQUENCE)} AA")
print(f"Toxin Probability: {score:.4f}")
print(f"Risk Level: {level}")
if score > 0.5:
    print("ðŸš¨ WARNING: High probability of toxic function. Flagged for review.")
else:
    print("âœ… SAFE: No toxic function detected.")

### Mechanistic Explanation
How did the model make this decision? We use Direct Logit Attribution (DLA) to see which layers contributed to the final score.

In [ ]:
# ── 3. Explain Decision (Layer Attribution) ──────────────────────────
explanation = explain(TEST_SEQUENCE, tok, esm2, probe_dir_np, sae=sae)
dpa = explanation['layer_attribution']
top_layers = explanation['top_discriminating_layers']

plt.figure(figsize=(10, 4))
colors = ['#E07B39' if val > 0 else '#2E86AB' for val in dpa]
plt.bar(range(1, 34), dpa, color=colors)
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Direct Logit Attribution: Which layers drove the decision?')
plt.xlabel('ESM-2 Layer')
plt.ylabel('Attribution Score')
plt.show()

print(f"Key layers driving this decision: {top_layers}")


### Motif Extraction
Visualizing the exact structural features driving the alarm.

In [ ]:
# ── 4. SAE Toxin Motif Extraction ────────────────────────────────────
if 'sae_active_toxin_features' in explanation and explanation['sae_active_toxin_features']:
    active_sae = explanation['sae_active_toxin_features']
    
    labels = list(active_sae.keys())
    vals = [feat['val'] for feat in active_sae.values()]
    types = [feat['type'] for feat in active_sae.values()]
    
    # Color map: Red for Robust (Adversarially exploited), Blue for Evadable
    color_map = {'ROBUST': '#D1495B', 'EVADABLE': '#2E86AB'}
    bar_colors = [color_map.get(t, '#888888') for t in types]
    
    # Append type to label for clarity
    display_labels = [f"[{t}] {l}" for l, t in zip(labels, types)]
    
    plt.figure(figsize=(10, len(labels)*0.4 + 1))
    plt.barh(display_labels, vals, color=bar_colors)
    plt.title('Sparse Autoencoder (SAE): Detected Structural Toxin Motifs')
    plt.xlabel('Feature Activation Strength')
    plt.gca().invert_yaxis()
    
    # Custom legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#D1495B', label='ROBUST (Core Scaffold)'),
                       Patch(facecolor='#2E86AB', label='EVADABLE (Surface Motif)')]
    plt.legend(handles=legend_elements, loc='lower right')
    
    plt.show()
elif sae is None:
    print("\n(Note: SAE not loaded. To see specific structural motifs, install interplm.)")
else:
    print("\nNo canonical toxin motifs detected by SAE.")


### Adversarial Defense
Using mechanistic interpretability to detect deceptive designs.

In [ ]:
# ── 5. Adversarial Double-Evader Detection ───────────────────────────
if 'sae_active_toxin_features' in explanation and explanation['sae_active_toxin_features']:
    active_sae = explanation['sae_active_toxin_features']
    has_robust = any(v['type'] == 'ROBUST' for v in active_sae.values())
    has_evadable = any(v['type'] == 'EVADABLE' for v in active_sae.values())
    
    if has_robust and not has_evadable:
        print("\n" + "="*70)
        print("🚨 DECEPTIVE SIGNATURE DETECTED: POTENTIAL DOUBLE-EVADER 🚨")
        print("="*70)
        print("The sequence strongly activates core structural scaffolds (ROBUST)")
        print("but completely lacks evolutionary surface motifs (EVADABLE).")
        print("\nThis is the hallmark signature of an adversarial AI redesign ")
        print("that attempts to bypass sequence-based screening tools.")
        print("="*70 + "\n")
    else:
        print("\n✅ Structural signature is consistent with natural evolution (Non-Deceptive).\n")
